# Review the best campaign trial

**Purpose:** Review the best campaign trial.

**Before you start:** Completed campaign trials and saved model artifacts. Use the Python environment prepared by [setup](../setup.ipynb).

**Results:** Trial ranking and generated molecule review.

Run the cells in order, reviewing the configuration before starting the main work. Data stays under `notebooks/datasets`; models and outputs use the project’s artifact folders.


Load the campaign review tools and locate saved results.


In [ ]:
print('Load the campaign review tools and locate saved results.')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

from IPython.display import display
import pandas as pd

from conditional_node_field_graph_generator.notebooks import configure_notebook
globals().update(configure_notebook(require_nsppk=True, print_torch=True))

from conditional_node_field_graph_generator.extensions.demo import (
    build_campaign_trial_artificial_plotter,
    collect_campaign_trial_results,
    draw_artificial_graphs,
    load_campaign_trial_generator,
    load_campaign_trial_training_examples,
    select_best_campaign_trial,
)
from conditional_node_field_graph_generator.extensions.demo.visualization import (
    infer_display_mode,
    plot_networkx_graphs,
    show_molecules,
)


Choose a campaign, or use the latest one, and select the compute device.


In [ ]:
print('Choose a campaign, or use the latest one, and select the compute device.')
CAMPAIGN_STATE_PATH = None  # keep None to auto-select the latest active campaign
DEVICE = "cpu"

Rank completed trials and select the best available result.


In [ ]:
print('Rank completed trials and select the best available result.')
ranking = collect_campaign_trial_results(
    repo_root=REPO_ROOT,
    campaign_state_path=CAMPAIGN_STATE_PATH,
)
selection = select_best_campaign_trial(
    repo_root=REPO_ROOT,
    campaign_state_path=CAMPAIGN_STATE_PATH,
)

print(f"Campaign: {selection.domain}/{selection.prefix}")
print(f"Best run: {selection.run_dir}")
print(f"Best trial: {selection.trial_dir.name}")
print(f"Checkpoint: {selection.checkpoint_path}")
display(ranking.head(20))


Load and display training examples for the selected trial.


In [ ]:
print('Load and display training examples for the selected trial.')
TRAINING_EXAMPLE_COUNT = 7

training_examples = load_campaign_trial_training_examples(
    selection,
    notebook_context=globals(),
    n_examples=TRAINING_EXAMPLE_COUNT,
)

example_titles = [f"train {index}" for index in range(1, len(training_examples) + 1)]
example_display_mode = infer_display_mode(training_examples)
artificial_plotter = None

if selection.domain == "artificial_graphs":
    artificial_plotter = build_campaign_trial_artificial_plotter(selection)
    draw_artificial_graphs(
        training_examples,
        n=len(training_examples),
        title="Training dataset examples",
        titles=example_titles,
        n_graphs_per_line=TRAINING_EXAMPLE_COUNT,
        plotter=artificial_plotter,
    )
elif example_display_mode == "molecule":
    show_molecules(
        training_examples,
        n=len(training_examples),
        title="Training dataset examples",
        legends=example_titles,
        n_graphs_per_line=TRAINING_EXAMPLE_COUNT,
    )
else:
    plot_networkx_graphs(
        training_examples,
        n_cols=TRAINING_EXAMPLE_COUNT,
        mode=example_display_mode,
        titles=example_titles,
    )


Load the selected trial’s saved generator.


In [ ]:
print('Load the selected trial’s saved generator.')
graph_generator = load_campaign_trial_generator(
    selection,
    notebook_context=globals(),
    device=DEVICE,
)

print(f"Loaded checkpoint: {selection.checkpoint_path}")
print("is_fitted_ =", getattr(graph_generator, "is_fitted_", None))


Choose the number of samples and feasibility settings.


In [ ]:
print('Choose the number of samples and feasibility settings.')
N_SAMPLES = 2
FEASIBILITY_EFFORT = 2
FEASIBILITY_FILTER = "none"  # "none", "fallback", or "strict"

MAX_DISPLAY_GRAPHS = 16
N_COLS = 4

Generate molecules from the selected saved model.


In [ ]:
print('Generate molecules from the selected saved model.')
generated_graphs = list(
    graph_generator.sample(
        n_samples=N_SAMPLES,
        feasibility_effort=FEASIBILITY_EFFORT,
        feasibility_filter=FEASIBILITY_FILTER,
    )
)

try:
    violation_counts = [
        float(value)
        for value in graph_generator.feasibility_estimator.number_of_violations(generated_graphs)
    ]
    violation_error = None
except SystemError as exc:
    violation_counts = [float("nan")] * len(generated_graphs)
    violation_error = f"{type(exc).__name__}: {exc}"
    print(f"Violation summary unavailable: {violation_error}")
sample_summary = {
    "n_samples": N_SAMPLES,
    "returned_samples": len(generated_graphs),
    "feasibility_effort": FEASIBILITY_EFFORT,
    "feasibility_filter": FEASIBILITY_FILTER,
    "average_num_violations": (
        sum(violation_counts) / len(violation_counts) if violation_counts else float("nan")
    ),
    "feasible_rate": (
        sum(value == 0 for value in violation_counts) / len(violation_counts)
        if violation_counts and violation_error is None
        else float("nan")
    ),
    "violation_error": violation_error,
}

display(pd.DataFrame([sample_summary]))


Display the generated molecules and their evaluation results.


In [ ]:
print('Display the generated molecules and their evaluation results.')
graphs_to_show = generated_graphs[:MAX_DISPLAY_GRAPHS]
titles = [
    "v=n/a" if pd.isna(value) else f"v={value:g}"
    for value in violation_counts[:len(graphs_to_show)]
]
display_mode = infer_display_mode(graphs_to_show)

if selection.domain == "artificial_graphs":
    artificial_plotter = artificial_plotter or build_campaign_trial_artificial_plotter(selection)
    draw_artificial_graphs(
        graphs_to_show,
        title="Best campaign model samples",
        titles=titles,
        n_graphs_per_line=N_COLS,
        plotter=artificial_plotter,
    )
elif display_mode == "molecule":
    show_molecules(
        graphs_to_show,
        n=len(graphs_to_show),
        title="Best campaign model samples",
        legends=titles,
        n_graphs_per_line=N_COLS,
    )
else:
    plot_networkx_graphs(
        graphs_to_show,
        n_cols=N_COLS,
        mode=display_mode,
        titles=titles,
    )
